# GroupDNA — Your WhatsApp Group Chat, Decoded

> *"Spotify Wrapped, but for your friend group."*

| Field | Value |
|---|---|
| **Name** | Patel Mohitkumar Dipakkumar |
| **Project** | GroupDNA — Minor Project |
| **Academy** | The Unlox Academy |
| **Date** | 25 June 2026 |


## Feature 1 : Chat Parser

In [1]:
# ==========================================
# Feature 1 : Chat Parser
# ==========================================
# Reads hostel_bois.txt line by line.
# Extracts: date, time, hour, sender, message.
# Handles: system messages, media omitted,
#          deleted messages, multi-line messages.
# Stores all real messages as a list of dicts.

import numpy as np
from datetime import datetime, timedelta


def read_chat_file(file_name):
    """Read the WhatsApp export file and return lines."""
    with open(file_name, "r", encoding="utf-8") as f:
        return f.readlines()


def parse_chat(lines):
    """
    Parse raw WhatsApp export lines into structured dicts.
    Returns (messages_list, system_count, media_count, deleted_count).
    Each message dict has keys: sender, date, time, hour, message.
    """
    chat_messages      = []
    system_msg_count   = 0
    media_msg_count    = 0
    deleted_msg_count  = 0
    previous_message   = None

    for line in lines:
        line = line.strip()

        # Skip empty lines
        if not line:
            continue

        # --- Check if line starts with a date pattern DD/MM/YY ---
        if len(line) >= 8 and line[2] == "/" and line[5] == "/":

            # Split into date and the rest
            try:
                date_part, rest = line.split(", ", 1)
                time_part, rest2 = rest.split(" - ", 1)
            except ValueError:
                continue

            # System message: no ": " after the dash
            if ": " not in rest2:
                system_msg_count += 1
                continue

            sender, message = rest2.split(": ", 1)
            hour = int(time_part.split(":")[0])

            # Tag media and deleted (still count as sender messages)
            if message == "<Media omitted>":
                media_msg_count += 1

            elif message == "This message was deleted":
                deleted_msg_count += 1

            current_message = {
                "sender"  : sender,
                "date"    : date_part,
                "time"    : time_part,
                "hour"    : hour,
                "message" : message,
            }

            chat_messages.append(current_message)
            previous_message = current_message

        else:
            # Multi-line continuation — append to previous message
            if previous_message is not None:
                previous_message["message"] += "\n" + line

    return (
        chat_messages,
        system_msg_count,
        media_msg_count,
        deleted_msg_count,
    )


# ------------------------------------------
# Run the parser
# ------------------------------------------

FILE_NAME = "hostel_bois.txt"

lines = read_chat_file(FILE_NAME)

(
    chat_messages,
    system_msg_count,
    media_msg_count,
    deleted_msg_count,
) = parse_chat(lines)

# Derived globals used by all later features
participants      = sorted(set(m["sender"] for m in chat_messages))
start_date        = datetime.strptime(chat_messages[0]["date"], "%d/%m/%y")
end_date          = datetime.strptime(chat_messages[-1]["date"], "%d/%m/%y")
total_days        = (end_date - start_date).days + 1
total_messages    = len(chat_messages)

# ------------------------------------------
# Parser summary
# ------------------------------------------

print("=" * 60)
print(" CHAT PARSER SUMMARY".center(60))
print("=" * 60)
print(f" Total Messages Parsed : {total_messages}")
print(f" System Messages       : {system_msg_count}")
print(f" Media Messages        : {media_msg_count}")
print(f" Deleted Messages      : {deleted_msg_count}")
print(f" Participants Found    : {len(participants)}")
print(f" Date Range            : {start_date.strftime('%d %B %Y')} to {end_date.strftime('%d %B %Y')}")
print(f" Duration              : {total_days} days")
print()
print(f" First message: {chat_messages[0]}")
print(f" Last  message: {chat_messages[-1]}")


                     CHAT PARSER SUMMARY                    
 Total Messages Parsed : 3174
 System Messages       : 4
 Media Messages        : 32
 Deleted Messages      : 15
 Participants Found    : 6
 Date Range            : 01 April 2024 to 30 May 2024
 Duration              : 60 days

 First message: {'sender': 'Rahul', 'date': '01/04/24', 'time': '01:17', 'hour': 1, 'message': 'scene fix'}
 Last  message: {'sender': 'Aman', 'date': '30/05/24', 'time': '23:31', 'hour': 23, 'message': 'anyone awake?'}


## Feature 2 : Group Overview

In [2]:
# ==========================================
# Feature 2 : Group Overview
# ==========================================
# Computes per-person message counts.
# Prints the executive summary section of the report.

# ------------------------------------------
# Message count per participant
# ------------------------------------------

participant_count = {}

for msg in chat_messages:
    sender = msg["sender"]
    participant_count[sender] = participant_count.get(sender, 0) + 1

# Sort highest to lowest
sorted_participants = sorted(
    participant_count.items(),
    key=lambda x: x[1],
    reverse=True,
)

# ------------------------------------------
# Print Group Overview
# ------------------------------------------

print("=" * 60)
print(" GROUP OVERVIEW".center(60))
print("=" * 60)
print(f"  Group      : Hostel Bois 4ever")
print(f"  Period     : {start_date.strftime('%d %B %Y')} to {end_date.strftime('%d %B %Y')} ({total_days} days)")
print(f"  Total msgs : {total_messages:,}")
print(f"  Participants: {len(participants)}")
print()
print("  MESSAGES PER PERSON")
print("  " + "-" * 55)

max_count = sorted_participants[0][1]

for person, count in sorted_participants:
    pct   = (count / total_messages) * 100
    bar_l = int((count / max_count) * 20)
    bar   = "█" * bar_l if bar_l > 0 else "."
    print(f"  {person:<8} {bar:<22} {count:>4}  ({pct:5.1f}%)")


                       GROUP OVERVIEW                       
  Group      : Hostel Bois 4ever
  Period     : 01 April 2024 to 30 May 2024 (60 days)
  Total msgs : 3,174
  Participants: 6

  MESSAGES PER PERSON
  -------------------------------------------------------
  Rahul    ████████████████████    953  ( 30.0%)
  Priya    ███████████████         718  ( 22.6%)
  Neha     █████████████           635  ( 20.0%)
  Aman     ██████████              490  ( 15.4%)
  Karan    ███████                 354  ( 11.2%)
  Vikas    .                        24  (  0.8%)


## Feature 3 : Most Active Day and Hour

In [3]:
# ==========================================
# Feature 3 : Most Active Day and Hour
# ==========================================

# Count messages per day
day_count = {}
for msg in chat_messages:
    day_count[msg["date"]] = day_count.get(msg["date"], 0) + 1

# Count messages per hour (across all days)
hour_count = {}
for msg in chat_messages:
    hour_count[msg["hour"]] = hour_count.get(msg["hour"], 0) + 1

# Find busiest day and hour
busiest_day          = max(day_count, key=day_count.get)
busiest_day_count    = day_count[busiest_day]

busiest_hour         = max(hour_count, key=hour_count.get)
busiest_hour_count   = hour_count[busiest_hour]

# Format busiest day to readable string
formatted_busiest_day = datetime.strptime(busiest_day, "%d/%m/%y").strftime("%d %B %Y")

# Avg messages per hour across 60 days
avg_per_hour = busiest_hour_count / total_days

print("=" * 60)
print(" MOST ACTIVE DAY & HOUR".center(60))
print("=" * 60)
print(f"  Busiest day  : {formatted_busiest_day} ({busiest_day_count} messages)")
print(f"  Busiest hour : {busiest_hour:02d}:00 - {(busiest_hour+1)%24:02d}:00  (avg {avg_per_hour:.0f} msgs/day)")


                   MOST ACTIVE DAY & HOUR                   
  Busiest day  : 04 May 2024 (76 messages)
  Busiest hour : 18:00 - 19:00  (avg 4 msgs/day)


## Feature 4 : Activity Heatmap (NumPy)

In [4]:
# ==========================================
# Feature 4 : Activity Heatmap (NumPy)
# ==========================================
# Builds a (participants x 24) NumPy matrix.
# Each cell = messages sent by that person in that hour.
# Renders as a text-art heatmap using block characters.
# Per-person shading: relative to that person's own max.

# ------------------------------------------
# Build participant index
# ------------------------------------------

# Use the PDF's expected order (by message count)
ordered_participants = [p for p, _ in sorted_participants]
participant_index    = {p: i for i, p in enumerate(ordered_participants)}

# ------------------------------------------
# Build 6 x 24 activity matrix with NumPy
# ------------------------------------------

activity_matrix = np.zeros((len(ordered_participants), 24), dtype=int)

for msg in chat_messages:
    row = participant_index[msg["sender"]]
    col = msg["hour"]
    activity_matrix[row, col] += 1

# ------------------------------------------
# Shading helper — relative to each person's own max
# ------------------------------------------

def get_shade(value, person_max):
    """Return block char based on value vs person's max."""
    if person_max == 0 or value == 0:
        return "."
    ratio = value / person_max
    if ratio <= 0.25:
        return "░"
    elif ratio <= 0.50:
        return "▒"
    else:
        return "█"

# ------------------------------------------
# Print heatmap
# ------------------------------------------

print("=" * 72)
print(" ACTIVITY HEATMAP (messages by hour)".center(72))
print("=" * 72)

# Header row — show every 3rd hour
print(f"  {'Name':<10}", end="")
for h in range(0, 24, 3):
    print(f"  {h:02d}", end="")
print()
print("  " + "-" * 66)

for i, person in enumerate(ordered_participants):
    row      = activity_matrix[i]
    per_max  = int(row.max())
    print(f"  {person:<10}", end="")
    for h in range(0, 24, 3):
        # Average the 3 hours in this block for display
        block_val = int(row[h:h+3].sum())
        shade     = get_shade(block_val, per_max * 3)
        print(f"  {shade} ", end="")
    # Tag the night owl
    night_frac = (int(row[23:24].sum()) + int(row[0:5].sum())) / (row.sum() or 1)
    if night_frac > 0.6:
        print("  <- NIGHT OWL", end="")
    print()

print()
print("  Legend:  .=none  ░=low  ▒=medium  █=high  (relative to each person)")


                   ACTIVITY HEATMAP (messages by hour)                  
  Name        00  03  06  09  12  15  18  21
  ------------------------------------------------------------------
  Rahul       ░   ░   ░   ░   ▒   █   █   █ 
  Priya       .   .   ▒   █   █   █   █   ▒ 
  Neha        .   ░   ▒   █   █   █   █   ▒ 
  Aman        █   █   .   .   ░   ░   ░   ▒   <- NIGHT OWL
  Karan       .   .   ░   ▒   █   █   █   ▒ 
  Vikas       .   .   ▒   ░   ▒   █   █   ▒ 

  Legend:  .=none  ░=low  ▒=medium  █=high  (relative to each person)


## Feature 5 : Top Words (Word Frequency)

In [5]:
# ==========================================
# Feature 5 : Top Words (Word Frequency)
# ==========================================
# Counts every word across all real messages.
# Excludes stop words and short tokens.
# Renders a bar chart using block characters.

# ------------------------------------------
# Stop-word list (common English + Hindi filler)
# ------------------------------------------

stop_words = set([
    "i","is","a","an","and","or","to","of","in","on","at","for",
    "with","this","that","me","my","we","our","it","you","he","she",
    "they","was","are","be","been","has","have","had","do","did","will",
    "would","could","should","not","from","but","so","if","as","by",
    "about","up","out","what","when","who","how","all","just","like",
    "get","got","its","the","am","were","his","her","their","there",
    "then","than","them","these","those","which","said","can","more",
    "no","yes","na","ok","okay","ha","hai","h","k","hi","go","come",
    "also","even","well","here","into","your","some","time","very",
    "because","after","before","back","each","much","only","same",
    "such","new","old","used","two","over","too","any","tell","told",
    "s","t","m","re","ll","d","ve","n","co",
])

# ------------------------------------------
# Build word frequency dict
# ------------------------------------------

word_freq = {}

for msg in chat_messages:
    # Skip media and deleted
    if msg["message"] in ("<Media omitted>", "This message was deleted"):
        continue

    text = msg["message"].lower()

    # Strip punctuation
    for ch in ",.!?;:\'\"()[]{}<>/-_":
        text = text.replace(ch, " ")

    for word in text.split():
        word = word.strip()
        if len(word) <= 1:
            continue
        if word in stop_words:
            continue
        if not any(c.isalpha() for c in word):
            continue
        word_freq[word] = word_freq.get(word, 0) + 1

# Sort descending
sorted_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)

# ------------------------------------------
# Print Top 10 with bar chart
# ------------------------------------------

print("=" * 60)
print(" THIS GROUP'S FAVOURITE WORDS".center(60))
print("=" * 60)
print()

top_count = sorted_words[0][1]

for word, freq in sorted_words[:10]:
    bar_len = int((freq / top_count) * 24)
    bar     = "█" * bar_len if bar_len > 0 else "░"
    print(f"  {word:<12} {bar:<25}  {freq}")


                THIS GROUP'S FAVOURITE WORDS                

  guys         ████████████████████████   318
  today        ██████████████████████     292
  everyone     ███████████████            203
  telling      █████████████              179
  bhai         ████████████               160
  anyone       ███████████                157
  one          ███████████                157
  started      ███████████                150
  scene        ██████████                 145
  entire       ██████████                 145


## Feature 6 : Response Speed & Silent Streaks

In [6]:
# ==========================================
# Feature 6 : Response Speed & Silent Streaks
# ==========================================
# (a) Average response time per person.
#     Gap = time between a msg from person A and
#     the very next msg from a DIFFERENT person B.
# (b) Longest consecutive silent streak per person.

# ------------------------------------------
# (a) Average Response Time
# ------------------------------------------

resp_gap   = {p: 0   for p in participants}
resp_count = {p: 0   for p in participants}

for i in range(1, len(chat_messages)):
    prev_msg = chat_messages[i - 1]
    curr_msg = chat_messages[i]

    # Only count when the sender changes
    if prev_msg["sender"] == curr_msg["sender"]:
        continue

    t_prev = datetime.strptime(
        prev_msg["date"] + " " + prev_msg["time"], "%d/%m/%y %H:%M"
    )
    t_curr = datetime.strptime(
        curr_msg["date"] + " " + curr_msg["time"], "%d/%m/%y %H:%M"
    )
    gap_seconds = (t_curr - t_prev).total_seconds()

    # Only positive gaps (no backward timestamps)
    if gap_seconds >= 0:
        resp_gap[curr_msg["sender"]]   += gap_seconds
        resp_count[curr_msg["sender"]] += 1

# Identify fastest and slowest
fastest_person = None
slowest_person = None
fastest_time   = float("inf")
slowest_time   = 0.0

avg_resp = {}

for person in participants:
    if resp_count[person] == 0:
        continue
    avg_min = (resp_gap[person] / resp_count[person]) / 60
    avg_resp[person] = avg_min
    if avg_min < fastest_time:
        fastest_time   = avg_min
        fastest_person = person
    if avg_min > slowest_time:
        slowest_time   = avg_min
        slowest_person = person

# ------------------------------------------
# Print Response Times
# ------------------------------------------

print("=" * 60)
print(" RESPONSE PATTERNS".center(60))
print("=" * 60)
print()

for person in sorted(avg_resp, key=avg_resp.get):
    avg_m = avg_resp[person]
    label = ""
    if person == fastest_person:
        label = "  <- FASTEST"
    elif person == slowest_person:
        label = "  <- SLOWEST"
    if avg_m < 60:
        time_str = f"{avg_m:.1f} min"
    else:
        time_str = f"{avg_m/60:.1f} hrs"
    print(f"  {person:<10} avg response : {time_str:<12}{label}")

print()
print(f"  Fastest replier : {fastest_person}  ({fastest_time:.1f} min)")
print(f"  Slowest replier : {slowest_person}  ({slowest_time/60:.1f} hrs)")

# ------------------------------------------
# (b) Longest Silent Streaks
# ------------------------------------------

# Build set of dates each person was active
active_date_sets = {p: set() for p in participants}
for msg in chat_messages:
    dt = datetime.strptime(msg["date"], "%d/%m/%y").date()
    active_date_sets[msg["sender"]].add(dt)

streak_data = {}   # person -> (longest_streak, streak_start, streak_end)

for person in participants:
    longest      = 0
    current      = 0
    current_start= None
    best_start   = None
    best_end     = None
    day          = start_date

    while day <= end_date:
        if day.date() in active_date_sets[person]:
            current       = 0
            current_start = None
        else:
            if current == 0:
                current_start = day
            current += 1
            if current > longest:
                longest    = current
                best_start = current_start
                best_end   = day
        day += timedelta(days=1)

    streak_data[person] = (longest, best_start, best_end)

# Sort by streak length descending
sorted_streaks = sorted(streak_data.items(), key=lambda x: x[1][0], reverse=True)

print()
print("=" * 60)
print(" LONGEST SILENT STREAKS".center(60))
print(" (consecutive days with zero messages)".center(60))
print("=" * 60)
print()

for person, (days, s, e) in sorted_streaks:
    if days == 0:
        note = "  (never went silent)"
        print(f"  {person:<10} : {days:>2} days{note}")
    elif s and e:
        print(f"  {person:<10} : {days:>2} days  ({s.strftime('%d %b')} to {e.strftime('%d %b')})")
    else:
        print(f"  {person:<10} : {days:>2} days")


                      RESPONSE PATTERNS                     

  Rahul      avg response : 34.9 min      <- FASTEST
  Karan      avg response : 36.6 min    
  Neha       avg response : 39.4 min    
  Priya      avg response : 42.0 min    
  Vikas      avg response : 46.3 min    
  Aman       avg response : 55.4 min      <- SLOWEST

  Fastest replier : Rahul  (34.9 min)
  Slowest replier : Aman  (0.9 hrs)

                   LONGEST SILENT STREAKS                   
            (consecutive days with zero messages)           

  Vikas      : 11 days  (23 Apr to 03 May)
  Aman       :  0 days  (never went silent)
  Karan      :  0 days  (never went silent)
  Neha       :  0 days  (never went silent)
  Priya      :  0 days  (never went silent)
  Rahul      :  0 days  (never went silent)


## Feature 7 : Personality Archetype Detection

In [7]:
# ==========================================
# Feature 7 : Personality Archetype Detection
# ==========================================
# For each person, computes a score for all 8 archetypes.
# Assigns the archetype with the HIGHEST score.
# Assignment is exclusive: once an archetype is claimed
# by the top scorer, it cannot be re-used.
# Tie-breaking rule: if two people have equal score on the
# same archetype, the one with more total messages wins.

# ------------------------------------------
# Compute raw scores for each archetype
# ------------------------------------------

# --- Score 1: THE SPAMMER ---
# Avg consecutive burst length (messages in a row without
# another person sending in between).
spam_burst  = {p: 0 for p in participants}
burst_count = {p: 0 for p in participants}

current_burst  = 1
current_sender = chat_messages[0]["sender"]

for i in range(1, len(chat_messages)):
    sender = chat_messages[i]["sender"]
    if sender == current_sender:
        current_burst += 1
    else:
        spam_burst[current_sender]  += current_burst
        burst_count[current_sender] += 1
        current_burst  = 1
        current_sender = sender

spam_burst[current_sender]  += current_burst
burst_count[current_sender] += 1

avg_burst_score = {
    p: (spam_burst[p] / burst_count[p]) if burst_count[p] else 0
    for p in participants
}

# --- Score 2: THE GROUP MOM ---
# Count of caring keyword hits across messages.
caring_keywords = [
    "okay","safe","eat","sleep","take care","are you","please",
    "reminder","drink water","don\'t forget","dont forget","care",
]

caring_score = {p: 0 for p in participants}

for msg in chat_messages:
    low = msg["message"].lower()
    for kw in caring_keywords:
        if kw in low:
            caring_score[msg["sender"]] += 1

# --- Score 3: THE NIGHT OWL ---
# % of messages sent between 23:00 and 04:59.
night_msgs  = {p: 0 for p in participants}
total_count = {p: 0 for p in participants}

for msg in chat_messages:
    total_count[msg["sender"]] += 1
    if msg["hour"] >= 23 or msg["hour"] <= 4:
        night_msgs[msg["sender"]] += 1

night_owl_score = {
    p: (night_msgs[p] / total_count[p] * 100) if total_count[p] else 0
    for p in participants
}

# --- Score 4: THE STORYTELLER ---
# Avg words per message (excluding media / deleted).
story_words  = {p: 0 for p in participants}
story_count  = {p: 0 for p in participants}

for msg in chat_messages:
    if msg["message"] in ("<Media omitted>", "This message was deleted"):
        continue
    story_words[msg["sender"]]  += len(msg["message"].split())
    story_count[msg["sender"]]  += 1

storyteller_score = {
    p: (story_words[p] / story_count[p]) if story_count[p] else 0
    for p in participants
}

# --- Score 5: THE DRAMA QUEEN ---
# % of messages where all alpha chars are uppercase
# OR message contains 2+ exclamation marks.
drama_count  = {p: 0 for p in participants}
drama_total  = {p: 0 for p in participants}

for msg in chat_messages:
    if msg["message"] in ("<Media omitted>", "This message was deleted"):
        continue
    text  = msg["message"]
    alpha = [c for c in text if c.isalpha()]
    drama_total[msg["sender"]] += 1
    if len(alpha) >= 3 and all(c.isupper() for c in alpha):
        drama_count[msg["sender"]] += 1
    elif text.count("!") >= 2:
        drama_count[msg["sender"]] += 1

drama_score = {
    p: (drama_count[p] / drama_total[p] * 100) if drama_total[p] else 0
    for p in participants
}

# --- Score 6: THE GHOST ---
# % of days in the chat period with zero messages.
ghost_score = {
    p: ((total_days - len(active_date_sets[p])) / total_days * 100)
    for p in participants
}

# --- Score 7: THE COMEDIAN ---
# Count of funny words (as % of messages).
funny_keywords = ["lol","lmao","haha","rofl","lmfao","hehe"]
funny_count = {p: 0 for p in participants}

for msg in chat_messages:
    low = msg["message"].lower()
    for kw in funny_keywords:
        if kw in low:
            funny_count[msg["sender"]] += 1

comedian_score = {
    p: (funny_count[p] / total_count[p] * 100) if total_count[p] else 0
    for p in participants
}

# --- Score 8: THE QUESTION MASTER ---
# % of messages ending with "?".
q_count = {p: 0 for p in participants}

for msg in chat_messages:
    if msg["message"].strip().endswith("?"):
        q_count[msg["sender"]] += 1

question_score = {
    p: (q_count[p] / total_count[p] * 100) if total_count[p] else 0
    for p in participants
}

# --- BONUS Score 9: THE PAKKA PUNCTUAL ---
# % of messages sent during daytime hours (09:00 to 18:00).
# Someone who messages only during sensible hours - no midnight chaos.
day_msgs = {p: 0 for p in participants}
for msg in chat_messages:
    if 9 <= msg["hour"] <= 18:
        day_msgs[msg["sender"]] += 1

punctual_score = {
    p: (day_msgs[p] / total_count[p] * 100) if total_count[p] else 0
    for p in participants
}

# ------------------------------------------
# Archetype assignment (exclusive)
# Tie-break: higher total message count wins.
# ------------------------------------------

archetype_scores = {
    "THE SPAMMER"        : avg_burst_score,
    "THE GROUP MOM"      : caring_score,
    "THE NIGHT OWL"      : night_owl_score,
    "THE STORYTELLER"    : storyteller_score,
    "THE DRAMA QUEEN"    : drama_score,
    "THE GHOST"          : ghost_score,
    "THE COMEDIAN"       : comedian_score,
    "THE QUESTION MASTER": question_score,
    "THE PAKKA PUNCTUAL" : punctual_score,
}

# Normalise all scores to 0-100 range for fair comparison
def normalise(score_dict):
    max_v = max(score_dict.values()) if score_dict.values() else 1
    if max_v == 0:
        return {p: 0 for p in score_dict}
    return {p: (v / max_v * 100) for p, v in score_dict.items()}

norm_scores = {
    archetype: normalise(scores)
    for archetype, scores in archetype_scores.items()
}

assigned_archetypes = {}   # person -> archetype
claimed_archetypes  = set()

# For each person, rank archetypes by normalised score descending
# then pick the highest unclaimed one.
person_score_matrix = {}
for person in participants:
    ranked = sorted(
        norm_scores.keys(),
        key=lambda a: (norm_scores[a][person], participant_count[person]),
        reverse=True,
    )
    person_score_matrix[person] = ranked

# Greedy assignment: iterate until everyone has an archetype.
# Priority given to the person with the highest score on their top archetype.
remaining = list(participants)
for _ in range(len(participants)):
    if not remaining:
        break
    # Find person whose top available archetype score is highest
    best_person   = None
    best_archetype = None
    best_val      = -1
    for person in remaining:
        for arch in person_score_matrix[person]:
            if arch not in claimed_archetypes:
                val = norm_scores[arch][person]
                if val > best_val:
                    best_val       = val
                    best_person    = person
                    best_archetype = arch
                break
    if best_person:
        assigned_archetypes[best_person] = best_archetype
        claimed_archetypes.add(best_archetype)
        remaining.remove(best_person)

# ------------------------------------------
# Print Personality Archetypes
# ------------------------------------------

print("=" * 65)
print(" PERSONALITY ARCHETYPES".center(65))
print("=" * 65)
print()

for person, arch in sorted(assigned_archetypes.items(),
                            key=lambda x: participant_count[x[0]],
                            reverse=True):
    # Build a short reason string
    if arch == "THE SPAMMER":
        reason = f"avg {avg_burst_score[person]:.1f} msgs in a row"
    elif arch == "THE GROUP MOM":
        reason = f"caring keyword score: {caring_score[person]}"
    elif arch == "THE NIGHT OWL":
        reason = f"{night_owl_score[person]:.1f}% msgs between 23h-04h"
    elif arch == "THE STORYTELLER":
        reason = f"avg {storyteller_score[person]:.1f} words per msg"
    elif arch == "THE DRAMA QUEEN":
        reason = f"{drama_score[person]:.1f}% ALL-CAPS messages"
    elif arch == "THE GHOST":
        silent_days = total_days - len(active_date_sets[person])
        reason = f"silent on {silent_days} of {total_days} days"
    elif arch == "THE COMEDIAN":
        reason = f"{funny_count[person]} funny-word hits"
    elif arch == "THE QUESTION MASTER":
        reason = f"{question_score[person]:.1f}% messages end with ?"
    elif arch == "THE PAKKA PUNCTUAL":
        reason = f"{punctual_score[person]:.1f}% msgs during 09-18h"
    else:
        reason = ""

    print(f"  {person:<8} -> {arch:<22}  ({reason})")

print()
print("  BONUS ARCHETYPE: THE PAKKA PUNCTUAL")
print("  Detection rule: highest % of messages sent during 09:00-18:00.")
print("  Represents the disciplined student who stays off-screen at night.")
print("  (AI-assisted: Claude helped structure the normalisation logic here.)")


                      PERSONALITY ARCHETYPES                     

  Rahul    -> THE SPAMMER             (avg 4.5 msgs in a row)
  Priya    -> THE GROUP MOM           (caring keyword score: 703)
  Neha     -> THE DRAMA QUEEN         (63.3% ALL-CAPS messages)
  Aman     -> THE NIGHT OWL           (79.8% msgs between 23h-04h)
  Karan    -> THE STORYTELLER         (avg 57.0 words per msg)
  Vikas    -> THE GHOST               (silent on 44 of 60 days)

  BONUS ARCHETYPE: THE PAKKA PUNCTUAL
  Detection rule: highest % of messages sent during 09:00-18:00.
  Represents the disciplined student who stays off-screen at night.
  (AI-assisted: Claude helped structure the normalisation logic here.)


## Feature 8 : Final Report

In [8]:
# ==========================================
# Feature 8 : FINAL REPORT
# ==========================================
# Assembles all features into one clean,
# formatted report you would actually screenshot.

print()
print("=" * 62)
print('  GROUPDNA REPORT — "Hostel Bois 4ever"'.center(62))
print(f"  {total_days} days  •  {total_messages:,} messages  •  {len(participants)} members".center(62))
print("=" * 62)
print()
print(f"  Period      : {start_date.strftime('%d %B %Y')} to {end_date.strftime('%d %B %Y')}")
print(f"  Busiest day : {formatted_busiest_day} ({busiest_day_count} messages)")
print(f"  Busiest hour: {busiest_hour:02d}:00 - {(busiest_hour+1)%24:02d}:00")
print()

# ---- MESSAGES PER PERSON ----
print("  " + "─" * 58)
print("  MESSAGES PER PERSON")
print("  " + "─" * 58)
max_count = sorted_participants[0][1]
for person, count in sorted_participants:
    pct   = (count / total_messages) * 100
    bar_l = int((count / max_count) * 22)
    bar   = "█" * bar_l if bar_l > 0 else "."
    print(f"  {person:<8} {bar:<23} {count:>4}  ({pct:5.1f}%)")

print()

# ---- ACTIVITY HEATMAP ----
print("  " + "─" * 58)
print("  ACTIVITY HEATMAP  (hour of day, columns 00 to 23)")
print("  " + "─" * 58)
print(f"  {'':<10}", end="")
for h in range(0, 24, 3):
    print(f" {h:02d}", end="")
print()
print("  " + "-" * 56)

for i, person in enumerate(ordered_participants):
    row     = activity_matrix[i]
    per_max = int(row.max())
    night_f = (int(row[23:24].sum()) + int(row[0:5].sum())) / (row.sum() or 1)
    print(f"  {person:<10}", end="")
    for h in range(0, 24, 3):
        block_val = int(row[h:h+3].sum())
        shade     = get_shade(block_val, per_max * 3)
        print(f"  {shade} ", end="")
    if night_f > 0.6:
        print("  <- NIGHT OWL", end="")
    print()

print()

# ---- TOP WORDS ----
print("  " + "─" * 58)
print("  THIS GROUP'S FAVOURITE WORDS")
print("  " + "─" * 58)
top_count_val = sorted_words[0][1]
for word, freq in sorted_words[:10]:
    bar_len = int((freq / top_count_val) * 22)
    bar     = "█" * bar_len if bar_len > 0 else "░"
    print(f"  {word:<12} {bar:<23} {freq}")

print()

# ---- RESPONSE PATTERNS ----
print("  " + "─" * 58)
print("  RESPONSE PATTERNS")
print("  " + "─" * 58)
print(f"  Fastest replier : {fastest_person}  (avg {fastest_time:.1f} minutes)")
slw_hrs = slowest_time / 60
print(f"  Slowest replier : {slowest_person}  (avg {slw_hrs:.1f} hours)")

print()

# ---- SILENT STREAKS ----
print("  " + "─" * 58)
print("  LONGEST SILENT STREAKS")
print("  " + "─" * 58)
for person, (days, s, e) in sorted_streaks:
    if days == 0:
        print(f"  {person:<10} : {days:>2} days  (never went silent)")
    elif s and e:
        print(f"  {person:<10} : {days:>2} days  ({s.strftime('%d %b')} to {e.strftime('%d %b')})")
    else:
        print(f"  {person:<10} : {days:>2} days")

print()

# ---- PERSONALITY ARCHETYPES ----
print("  " + "─" * 58)
print("  PERSONALITY ARCHETYPES")
print("  " + "─" * 58)

for person, arch in sorted(assigned_archetypes.items(),
                            key=lambda x: participant_count[x[0]],
                            reverse=True):
    if arch == "THE SPAMMER":
        reason = f"avg {avg_burst_score[person]:.1f} msgs in a row"
    elif arch == "THE GROUP MOM":
        reason = f"caring keyword score: {caring_score[person]}"
    elif arch == "THE NIGHT OWL":
        reason = f"{night_owl_score[person]:.1f}% msgs between 23h-04h"
    elif arch == "THE STORYTELLER":
        reason = f"avg {storyteller_score[person]:.1f} words per msg"
    elif arch == "THE DRAMA QUEEN":
        reason = f"{drama_score[person]:.1f}% ALL-CAPS messages"
    elif arch == "THE GHOST":
        silent_days = total_days - len(active_date_sets[person])
        reason = f"silent on {silent_days} of {total_days} days"
    elif arch == "THE COMEDIAN":
        reason = f"{funny_count[person]} funny-word hits"
    elif arch == "THE QUESTION MASTER":
        reason = f"{question_score[person]:.1f}% messages end with ?"
    elif arch == "THE PAKKA PUNCTUAL":
        reason = f"{punctual_score[person]:.1f}% msgs during 09-18h"
    else:
        reason = ""
    print(f"  {person:<8} -> {arch:<22}  ({reason})")

print()
print("=" * 62)
print("  Generated by GroupDNA  •  Built with Python + NumPy".center(62))
print("  Created by : Mohit Patel.".center(62))
print("=" * 62)



             GROUPDNA REPORT — "Hostel Bois 4ever"            
            60 days  •  3,174 messages  •  6 members          

  Period      : 01 April 2024 to 30 May 2024
  Busiest day : 04 May 2024 (76 messages)
  Busiest hour: 18:00 - 19:00

  ──────────────────────────────────────────────────────────
  MESSAGES PER PERSON
  ──────────────────────────────────────────────────────────
  Rahul    ██████████████████████   953  ( 30.0%)
  Priya    ████████████████         718  ( 22.6%)
  Neha     ██████████████           635  ( 20.0%)
  Aman     ███████████              490  ( 15.4%)
  Karan    ████████                 354  ( 11.2%)
  Vikas    .                         24  (  0.8%)

  ──────────────────────────────────────────────────────────
  ACTIVITY HEATMAP  (hour of day, columns 00 to 23)
  ──────────────────────────────────────────────────────────
             00 03 06 09 12 15 18 21
  --------------------------------------------------------
  Rahul       ░   ░   ░   ░   ▒   █   █